# Train, Tune, and Assess Thin-Cloud Classifiers

This notebook runs the structured training pipeline for the three final models: Decision Tree, Random Forest, and XGBoost. The reusable logic lives in `src/lswt_cloud_masking`; this notebook is only for configuration, execution, and quick inspection of outputs.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

from lswt_cloud_masking.model_training import TrainingConfig, run_training_pipeline

C:\Users\airani\.conda\envs\py8tsec\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configure

The default paths mirror the old merged train/test folder. Adjust `config` below if your CSVs or output folder are elsewhere.

In [4]:
config = TrainingConfig.from_json(ROOT / "configs" / "training_config.example.json")

# For a smoke test, uncomment these three lines before running the full Optuna search.
# config.n_trials_dt = 2
# config.n_trials_rf = 2
# config.n_trials_xgb = 2

config

TrainingConfig(train_csv='D:/Trishna/Landsat_processing/processed/train_test_all_merged/df_train_all.csv', test_csv='D:/Trishna/Landsat_processing/processed/train_test_all_merged/df_test_all.csv', output_dir='models/general', label_column='lst_class', drop_columns=['lakeN', 'lst_raw', 'lst_filt', 'raamap', 'vzamap', 'szamap'], random_state=42, cv_splits=5, scoring='accuracy', n_trials_dt=80, n_trials_rf=120, n_trials_xgb=120, optuna_n_jobs=1)

## Run Training

This writes tuned models, Optuna studies, reports, confusion matrices, and metadata under `config.output_dir`.

In [ ]:
result = run_training_pipeline(config)
result.keys()

[I 2026-08-03 10:08:10,463] A new study created in memory with name: DecisionTree_Optimization
[I 2026-08-03 10:08:16,921] Trial 0 finished with value: 0.6803769937167714 and parameters: {'max_depth': 14, 'min_samples_split': 14, 'min_samples_leaf': 10, 'criterion': 'log_loss', 'splitter': 'best'}. Best is trial 0 with value: 0.6803769937167714.
[I 2026-08-03 10:08:21,267] Trial 1 finished with value: 0.7204929917834704 and parameters: {'max_depth': 18, 'min_samples_split': 17, 'min_samples_leaf': 3, 'criterion': 'gini', 'splitter': 'best'}. Best is trial 1 with value: 0.7204929917834704.
[I 2026-08-03 10:08:23,679] Trial 2 finished with value: 0.5643305944900918 and parameters: {'max_depth': 4, 'min_samples_split': 20, 'min_samples_leaf': 10, 'criterion': 'entropy', 'splitter': 'best'}. Best is trial 1 with value: 0.7204929917834704.
[I 2026-08-03 10:08:24,046] Trial 3 finished with value: 0.6840502658289027 and parameters: {'max_depth': 18, 'min_samples_split': 10, 'min_samples_leaf'

In [ ]:
import pandas as pd

summary = pd.DataFrame({
    name: {
        "test_accuracy": metrics["test_accuracy"],
        "test_balanced_accuracy": metrics["test_balanced_accuracy"],
        "cv_mean": metrics["cv_mean"],
        "cv_std": metrics["cv_std"],
    }
    for name, metrics in result["metrics"].items()
}).T
summary

The saved model paths are recorded in `model_metadata.json`. Use the RF and XGBoost paths in the single-scene and batch masking pipelines.